# 判别分析实现 - LDA、QDA与贝叶斯分类器

## 目标

深入理解判别分析方法和贝叶斯分类器的原理与应用。

- 理解线性判别分析（LDA）的数学原理
- 理解二次判别分析（QDA）的区别
- 实现高斯朴素贝叶斯分类器
- 对比不同方法的性能
- 可视化决策边界

## 1. 数据准备

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_classification, make_blobs, load_wine
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import StandardScaler
import pandas as pd
import seaborn as sns

np.random.seed(42)

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("环境准备完成")

## 2. 数学原理

In [ ]:
class BayesClassifier:
    """
    贝叶斯分类器的基础实现
    
    基于贝叶斯定理：
    P(y|x) = P(x|y) * P(y) / P(x)
    
    其中：
    - P(y|x): 后验概率（我们要计算的）
    - P(x|y): 似然函数
    - P(y): 先验概率
    - P(x): 证据
    """
    
    def __init__(self):
        self.classes_ = None
        self.priors_ = None
        self.means_ = None
        self.covariances_ = None
    
    def _gaussian_pdf(self, x, mean, cov):
        """
    计算多元高斯分布的概率密度函数
    
    PDF(x) = (1 / sqrt((2π)^k |Σ|)) * exp(-0.5 * (x-μ)^T Σ^(-1) (x-μ))
    """
        k = len(mean)
        x_centered = x - mean
        
        # 计算行列式和逆矩阵
        det = np.linalg.det(cov)
        inv_cov = np.linalg.inv(cov)
        
        if det <= 0:
            # 处理奇异矩阵
            det = np.linalg.det(cov + np.eye(k) * 1e-6)
            inv_cov = np.linalg.inv(cov + np.eye(k) * 1e-6)
        
        # 计算马氏距离
        mahalanobis = x_centered @ inv_cov @ x_centered.T
        
        # 计算概率密度
        coeff = 1.0 / np.sqrt((2 * np.pi) ** k * det)
        pdf = coeff * np.exp(-0.5 * mahalanobis)
        
        return pdf
    
    def fit(self, X, y):
        """
    训练贝叶斯分类器
    """
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]
        
        # 计算先验概率
        self.priors_ = np.zeros(n_classes)
        self.means_ = np.zeros((n_classes, n_features))
        self.covariances_ = np.zeros((n_classes, n_features, n_features))
        
        for i, c in enumerate(self.classes_):
            mask = (y == c)
            X_class = X[mask]
            
            # 先验概率
            self.priors_[i] = len(X_class) / len(X)
            
            # 均值
            self.means_[i] = X_class.mean(axis=0)
            
            # 协方差矩阵
            self.covariances_[i] = np.cov(X_class.T)
        
        return self
    
    def predict(self, X):
        """
    预测类别
    """
        posteriors = np.zeros((X.shape[0], len(self.classes_)))
        
        for i in range(len(self.classes_)):
            for j, x in enumerate(X):
                # 计算 P(x|y) * P(y)
                likelihood = self._gaussian_pdf(x, self.means_[i], self.covariances_[i])
                posteriors[j, i] = likelihood * self.priors_[i]
        
        # 返回后验概率最大的类别
        return self.classes_[np.argmax(posteriors, axis=1)]
    
    def predict_proba(self, X):
        """
    预测概率
    """
        posteriors = np.zeros((X.shape[0], len(self.classes_)))
        
        for i in range(len(self.classes_)):
            for j, x in enumerate(X):
                likelihood = self._gaussian_pdf(x, self.means_[i], self.covariances_[i])
                posteriors[j, i] = likelihood * self.priors_[i]
        
        # 归一化
        sum_posterior = posteriors.sum(axis=1, keepdims=True)
        return posteriors / (sum_posterior + 1e-10)

print("贝叶斯分类器类定义完成")

## 3. 生成测试数据

In [ ]:
# 生成不同类型的数据集
def generate_datasets():
    datasets = {}
    
    # 数据集1：相同协方差（适合LDA）
    X_same_cov, y_same_cov = make_blobs(
        n_samples=300, n_features=2, centers=3,
        cluster_std=1.5, random_state=42
    )
    datasets['same_cov'] = (X_same_cov, y_same_cov, '相同协方差')
    
    # 数据集2：不同协方差（适合QDA）
    X_diff_cov1, _ = make_blobs(
        n_samples=100, n_features=2, centers=[[0, 0]],
        cluster_std=[[0.5, 1]], random_state=42
    )
    X_diff_cov2, _ = make_blobs(
        n_samples=100, n_features=2, centers=[[3, 3]],
        cluster_std=[[2, 0.5]], random_state=42
    )
    X_diff_cov3, _ = make_blobs(
        n_samples=100, n_features=2, centers=[[-3, 3]],
        cluster_std=1.0, random_state=42
    )
    X_diff_cov = np.vstack([X_diff_cov1, X_diff_cov2, X_diff_cov3])
    y_diff_cov = np.array([0]*100 + [1]*100 + [2]*100)
    datasets['diff_cov'] = (X_diff_cov, y_diff_cov, '不同协方差')
    
    # 数据集3：线性可分
    X_linear, y_linear = make_classification(
        n_samples=300, n_features=2, n_redundant=0,
        n_informative=2, n_clusters_per_class=1, random_state=42
    )
    datasets['linear'] = (X_linear, y_linear, '线性可分')
    
    # 数据集4：各向异性（旋转后的数据）
    X_aniso, y_aniso = make_blobs(
        n_samples=300, n_features=2, centers=3,
        cluster_std=1.0, random_state=42
    )
    transformation = [[0.6, -0.6], [-0.4, 0.8]]
    X_aniso = np.dot(X_aniso, transformation)
    datasets['aniso'] = (X_aniso, y_aniso, '各向异性')
    
    return datasets

datasets = generate_datasets()

print("生成的数据集:")
for name, (X, y, desc) in datasets.items():
    print(f"  {name}: {X.shape} - {desc}")

In [ ]:
# 可视化数据集
plt.figure(figsize=(14, 4))

for i, (name, (X, y, desc)) in enumerate(datasets.items()):
    plt.subplot(1, 4, i+1)
    
    colors = ['red', 'blue', 'green']
    for j, class_name in enumerate(np.unique(y)):
        mask = (y == class_name)
        plt.scatter(X[mask, 0], X[mask, 1], c=colors[j % len(colors)], alpha=0.6, s=30)
    
    plt.xlabel('特征 1')
    plt.ylabel('特征 2')
    plt.title(f'{desc}\n({name})')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. LDA vs QDA 对比

In [ ]:
# 在相同协方差数据上对比 LDA 和 QDA
X_same, y_same, desc_same = datasets['same_cov']

# 训练 LDA 和 QDA
lda = LinearDiscriminantAnalysis()
qda = QuadraticDiscriminantAnalysis()
naive_bayes = GaussianNB()

lda.fit(X_same, y_same)
qda.fit(X_same, y_same)
naive_bayes.fit(X_same, y_same)

print("相同协方差数据集上的准确率:")
print(f"  LDA: {lda.score(X_same, y_same):.4f}")
print(f"  QDA: {qda.score(X_same, y_same):.4f}")
print(f"  GaussianNB: {naive_bayes.score(X_same, y_same):.4f}")

# 在不同协方差数据上对比
X_diff, y_diff, desc_diff = datasets['diff_cov']

lda_diff = LinearDiscriminantAnalysis()
qda_diff = QuadraticDiscriminantAnalysis()
naive_bayes_diff = GaussianNB()

lda_diff.fit(X_diff, y_diff)
qda_diff.fit(X_diff, y_diff)
naive_bayes_diff.fit(X_diff, y_diff)

print(f"\n不同协方差数据集上的准确率:")
print(f"  LDA: {lda_diff.score(X_diff, y_diff):.4f}")
print(f"  QDA: {qda_diff.score(X_diff, y_diff):.4f}")
print(f"  GaussianNB: {naive_bayes_diff.score(X_diff, y_diff):.4f}")

## 5. 可视化决策边界

In [ ]:
# 绘制决策边界的函数
def plot_decision_boundary(model, X, y, title, ax=None):
    """
    绘制分类器的决策边界
    """
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    if hasattr(model, 'predict_proba'):
        Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
        Z = Z.reshape(xx.shape)
        levels = np.linspace(0, 1, 11)
        cmap = plt.cm.RdYlBu
    else:
        Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)
        levels = np.arange(len(np.unique(y)) + 1) - 0.5
        cmap = plt.cm.RdYlBu
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    
    # 绘制决策区域
    contour = ax.contourf(xx, yy, Z, levels=levels, cmap=cmap, alpha=0.3)
    
    # 绘制决策边界
    if hasattr(model, 'predict_proba'):
        ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    else:
        ax.contour(xx, yy, Z, levels=[-0.5, 0.5, 1.5], colors='black', linewidths=1)
    
    # 绘制数据点
    colors = ['red', 'blue', 'green']
    for i, class_name in enumerate(np.unique(y)):
        mask = (y == class_name)
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[i % len(colors)], 
                  alpha=0.7, s=50, edgecolors='black', linewidths=0.5)
    
    ax.set_xlabel('特征 1')
    ax.set_ylabel('特征 2')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    
    return ax

# 对比三种方法的决策边界
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 相同协方差数据
models_same = [('LDA', lda), ('QDA', qda), ('GaussianNB', naive_bayes)]
for i, (name, model) in enumerate(models_same):
    plot_decision_boundary(model, X_same, y_same, f'{name}\n准确率: {model.score(X_same, y_same):.3f}', axes[0, i])

# 不同协方差数据
models_diff = [('LDA', lda_diff), ('QDA', qda_diff), ('GaussianNB', naive_bayes_diff)]
for i, (name, model) in enumerate(models_diff):
    plot_decision_boundary(model, X_diff, y_diff, f'{name}\n准确率: {model.score(X_diff, y_diff):.3f}', axes[1, i])

axes[0, 0].set_ylabel('特征 2\n相同协方差')
axes[1, 0].set_ylabel('特征 2\n不同协方差')

plt.tight_layout()
plt.show()

## 6. 理解 LDA 和 QDA 的区别

In [ ]:
# 分析 LDA 和 QDA 的参数
print("LDA 参数:")
print(f"  类别数: {len(lda.classes_)}")
print(f"  共享协方差矩阵形状: {lda.covariance_.shape}")
print(f"  各类别均值:")
for i, (mean, class_name) in enumerate(zip(lda.means_, lda.classes_)):
    print(f"    类别 {class_name}: [{mean[0]:.2f}, {mean[1]:.2f}]")

print("\nQDA 参数:")
print(f"  类别数: {len(qda.classes_)}")
print(f"  各类别协方差矩阵形状: {qda.covariance_[0].shape}")
print(f"  各类别均值:")
for i, (mean, class_name) in enumerate(zip(qda.means_, qda.classes_)):
    print(f"    类别 {class_name}: [{mean[0]:.2f}, {mean[1]:.2f}]")

print("\n参数数量对比:")
n_classes = len(lda.classes_)
n_features = X_same.shape[1]

lda_params = n_classes * n_features + n_features * n_features  # 均值 + 共享协方差
qda_params = n_classes * n_features + n_classes * n_features * n_features  # 均值 + 各类协方差

print(f"  LDA 参数: {lda_params}")
print(f"  QDA 参数: {qda_params}")
print(f"  QDA/LDA 参数比: {qda_params/lda_params:.2f}")

## 7. 在真实数据集上的应用 - Iris

In [ ]:
# 加载 Iris 数据集
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
feature_names_iris = iris.feature_names
target_names_iris = iris.target_names

# 划分训练集和测试集
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

print(f"Iris 数据集:")
print(f"  样本数: {len(X_iris)}")
print(f"  特征数: {len(feature_names_iris)}")
print(f"  类别数: {len(target_names_iris)}")
print(f"  类别名称: {target_names_iris}")
print(f"  训练集: {X_train_iris.shape}")
print(f"  测试集: {X_test_iris.shape}")

In [ ]:
# 在 Iris 数据集上对比三种方法
methods = {
    'LDA': LinearDiscriminantAnalysis(),
    'QDA': QuadraticDiscriminantAnalysis(),
    'GaussianNB': GaussianNB()
}

results_iris = []

for name, model in methods.items():
    model.fit(X_train_iris, y_train_iris)
    
    train_score = model.score(X_train_iris, y_train_iris)
    test_score = model.score(X_test_iris, y_test_iris)
    cv_scores = cross_val_score(model, X_iris, y_iris, cv=5)
    
    results_iris.append({
        '方法': name,
        '训练集准确率': train_score,
        '测试集准确率': test_score,
        '交叉验证准确率': cv_scores.mean(),
        '交叉验证标准差': cv_scores.std()
    })

# 显示结果
results_iris_df = pd.DataFrame(results_iris)
print("Iris 数据集上的性能对比:")
print(results_iris_df.round(4).to_string(index=False))

## 8. LDA 降维可视化

In [ ]:
# 使用 LDA 降维到 2D
lda_2d = LinearDiscriminantAnalysis(n_components=2)
X_iris_2d = lda_2d.fit_transform(X_iris, y_iris)

print(f"LDA 降维:")
print(f"  原始维度: {X_iris.shape[1]}")
print(f"  降维后维度: {X_iris_2d.shape[1]}")
print(f"  解释方差比例: {lda_2d.explained_variance_ratio_.round(4)}")
print(f"  累计解释方差: {lda_2d.explained_variance_ratio_.sum():.4f}")

In [ ]:
# 可视化 LDA 降维结果
plt.figure(figsize=(12, 6))

# 原始数据（使用前两个特征）
plt.subplot(1, 2, 1)
colors = ['red', 'green', 'blue']
for i, target_name in enumerate(target_names_iris):
    mask = (y_iris == i)
    plt.scatter(X_iris[mask, 0], X_iris[mask, 1],
                c=colors[i], label=target_name, alpha=0.7, s=50)

plt.xlabel(feature_names_iris[0])
plt.ylabel(feature_names_iris[1])
plt.title('原始数据（前两个特征）')
plt.legend()
plt.grid(True, alpha=0.3)

# LDA 降维后的数据
plt.subplot(1, 2, 2)
for i, target_name in enumerate(target_names_iris):
    mask = (y_iris == i)
    plt.scatter(X_iris_2d[mask, 0], X_iris_2d[mask, 1],
                c=colors[i], label=target_name, alpha=0.7, s=50)

plt.xlabel(f'LD1 ({lda_2d.explained_variance_ratio_[0]:.2%} 方差)')
plt.ylabel(f'LD2 ({lda_2d.explained_variance_ratio_[1]:.2%} 方差)')
plt.title('LDA 降维（2D）')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. 混淆矩阵对比

In [ ]:
# 计算混淆矩阵
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (name, model) in enumerate(methods.items()):
    y_pred = model.predict(X_test_iris)
    cm = confusion_matrix(y_test_iris, y_pred)
    
    im = axes[i].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    axes[i].set_title(f'{name}')
    
    tick_marks = np.arange(len(target_names_iris))
    axes[i].set_xticks(tick_marks)
    axes[i].set_yticks(tick_marks)
    axes[i].set_xticklabels(target_names_iris)
    axes[i].set_yticklabels(target_names_iris)
    axes[i].set_xlabel('预测标签')
    axes[i].set_ylabel('真实标签')
    
    # 添加数值
    for r in range(cm.shape[0]):
        for c in range(cm.shape[1]):
            axes[i].text(c, r, format(cm[r, c], 'd'),
                     horizontalalignment="center",
                     color="white" if cm[r, c] > cm.max() / 2 else "black")
    
    plt.colorbar(im, ax=axes[i])

plt.tight_layout()
plt.show()

# 打印分类报告
for name, model in methods.items():
    y_pred = model.predict(X_test_iris)
    print(f"\n{name} 分类报告:")
    print(classification_report(y_test_iris, y_pred, target_names=target_names_iris, digits=3))

## 10. Wine 数据集实验

In [ ]:
# 加载 Wine 数据集
wine = load_wine()
X_wine = wine.data
y_wine = wine.target
feature_names_wine = wine.feature_names
target_names_wine = wine.target_names

# 标准化数据
scaler_wine = StandardScaler()
X_wine_scaled = scaler_wine.fit_transform(X_wine)

# 划分训练集和测试集
X_train_wine, X_test_wine, y_train_wine, y_test_wine = train_test_split(
    X_wine_scaled, y_wine, test_size=0.3, random_state=42, stratify=y_wine
)

print(f"Wine 数据集:")
print(f"  样本数: {len(X_wine)}")
print(f"  特征数: {len(feature_names_wine)}")
print(f"  类别数: {len(target_names_wine)}")

In [ ]:
# 在 Wine 数据集上对比
results_wine = []

for name, model_class in [('LDA', LinearDiscriminantAnalysis), 
                          ('QDA', QuadraticDiscriminantAnalysis),
                          ('GaussianNB', GaussianNB)]:
    model = model_class()
    
    try:
        model.fit(X_train_wine, y_train_wine)
        
        train_score = model.score(X_train_wine, y_train_wine)
        test_score = model.score(X_test_wine, y_test_wine)
        cv_scores = cross_val_score(model, X_wine_scaled, y_wine, cv=5)
        
        results_wine.append({
            '方法': name,
            '训练集准确率': train_score,
            '测试集准确率': test_score,
            '交叉验证准确率': cv_scores.mean(),
            '交叉验证标准差': cv_scores.std()
        })
    except Exception as e:
        print(f"{name} 错误: {e}")

# 显示结果
if results_wine:
    results_wine_df = pd.DataFrame(results_wine)
    print("\nWine 数据集上的性能对比:")
    print(results_wine_df.round(4).to_string(index=False))

## 11. 参数调优

In [ ]:
# LDA 参数调优（主要是正则化）
param_grid_lda = {
    'solver': ['svd', 'lsqr', 'eigen'],
    'shrinkage': [None, 'auto', 0.1, 0.5, 0.9]
}

# 注意：只有 'lsqr' 和 'eigen' 求解器支持 shrinkage
grid_search_lda = GridSearchCV(
    LinearDiscriminantAnalysis(),
    param_grid_lda,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search_lda.fit(X_train_iris, y_train_iris)

print("LDA 最佳参数:")
for param, value in grid_search_lda.best_params_.items():
    print(f"  {param}: {value}")
print(f"最佳交叉验证得分: {grid_search_lda.best_score_:.4f}")
print(f"测试集准确率: {grid_search_lda.score(X_test_iris, y_test_iris):.4f}")

In [ ]:
# QDA 参数调优（主要是正则化）
param_grid_qda = {
    'reg_param': [0.0, 0.1, 0.2, 0.5, 0.8, 1.0]
}

grid_search_qda = GridSearchCV(
    QuadraticDiscriminantAnalysis(),
    param_grid_qda,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

try:
    grid_search_qda.fit(X_train_iris, y_train_iris)
    
    print("\nQDA 最佳参数:")
    for param, value in grid_search_qda.best_params_.items():
        print(f"  {param}: {value}")
    print(f"最佳交叉验证得分: {grid_search_qda.best_score_:.4f}")
    print(f"测试集准确率: {grid_search_qda.score(X_test_iris, y_test_iris):.4f}")
except Exception as e:
    print(f"QDA 调优错误: {e}")

## 12. 正则化的影响

In [ ]:
# 测试不同 shrinkage 值对 LDA 的影响
shrinkage_values = [None, 'auto', 0.1, 0.3, 0.5, 0.7, 0.9]
train_scores_shrink = []
test_scores_shrink = []

for shrink in shrinkage_values:
    try:
        lda_temp = LinearDiscriminantAnalysis(solver='lsqr', shrinkage=shrink)
        lda_temp.fit(X_train_iris, y_train_iris)
        
        train_scores_shrink.append(lda_temp.score(X_train_iris, y_train_iris))
        test_scores_shrink.append(lda_temp.score(X_test_iris, y_test_iris))
    except Exception as e:
        train_scores_shrink.append(np.nan)
        test_scores_shrink.append(np.nan)

# 可视化
plt.figure(figsize=(10, 6))
shrink_labels = [str(s) if s is not None else 'None' for s in shrinkage_values]
plt.plot(shrink_labels, train_scores_shrink, 'b-o', label='训练集', linewidth=2, markersize=6)
plt.plot(shrink_labels, test_scores_shrink, 'r-o', label='测试集', linewidth=2, markersize=6)
plt.xlabel('Shrinkage 参数')
plt.ylabel('准确率')
plt.title('Shrinkage 对 LDA 性能的影响')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Shrinkage 参数说明:")
print("- None: 不使用正则化（默认）")
print("- 'auto': 自动选择正则化参数")
print("- 0.0-1.0: 手动设置正则化强度")
print("\nShrinkage 的作用:")
print("- 处理小样本或高维数据")
print("- 防止协方差矩阵奇异")
print("- 类似岭回归的正则化")

## 13. 样本量对性能的影响

In [ ]:
# 测试不同训练样本大小对 LDA 和 QDA 的影响
sample_sizes = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

lda_sample_scores = []
qda_sample_scores = []
nb_sample_scores = []

for size in sample_sizes:
    # 使用该比例的训练数据
    X_train_small, _, y_train_small, _ = train_test_split(
        X_iris, y_iris, train_size=size, random_state=42, stratify=y_iris
    )
    
    # 训练模型
    lda_sample = LinearDiscriminantAnalysis()
    qda_sample = QuadraticDiscriminantAnalysis()
    nb_sample = GaussianNB()
    
    lda_sample.fit(X_train_small, y_train_small)
    qda_sample.fit(X_train_small, y_train_small)
    nb_sample.fit(X_train_small, y_train_small)
    
    # 在完整测试集上评估
    lda_sample_scores.append(lda_sample.score(X_test_iris, y_test_iris))
    qda_sample_scores.append(qda_sample.score(X_test_iris, y_test_iris))
    nb_sample_scores.append(nb_sample.score(X_test_iris, y_test_iris))

# 可视化
plt.figure(figsize=(10, 6))
plt.plot([s*100 for s in sample_sizes], lda_sample_scores, 'b-o', label='LDA', linewidth=2, markersize=6)
plt.plot([s*100 for s in sample_sizes], qda_sample_scores, 'r-s', label='QDA', linewidth=2, markersize=6)
plt.plot([s*100 for s in sample_sizes], nb_sample_scores, 'g-^', label='GaussianNB', linewidth=2, markersize=6)
plt.xlabel('训练样本比例 (%)')
plt.ylabel('测试集准确率')
plt.title('训练样本量对性能的影响')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("观察:")
print("- 小样本时：LDA 通常比 QDA 稳定（参数少）")
print("- 大样本时：QDA 可能超过 LDA（灵活性强）")
print("- GaussianNB: 假设特征独立，在某些情况下表现最好")

## 14. 贝叶斯分类器与朴素贝叶斯

In [ ]:
# 实现朴素贝叶斯分类器
class NaiveBayesClassifier:
    """
    高斯朴素贝叶斯分类器实现
    
    朴素贝叶斯假设：特征之间相互独立
    """
    
    def __init__(self):
        self.classes_ = None
        self.priors_ = None
        self.means_ = None
        self.vars_ = None
    
    def fit(self, X, y):
        """
    训练朴素贝叶斯分类器
    """
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]
        
        self.priors_ = np.zeros(n_classes)
        self.means_ = np.zeros((n_classes, n_features))
        self.vars_ = np.zeros((n_classes, n_features))
        
        for i, c in enumerate(self.classes_):
            mask = (y == c)
            X_class = X[mask]
            
            self.priors_[i] = len(X_class) / len(X)
            self.means_[i] = X_class.mean(axis=0)
            self.vars_[i] = X_class.var(axis=0) + 1e-9  # 添加小常数避免除零
        
        return self
    
    def predict(self, X):
        """
    预测类别
    """
        log_posteriors = np.zeros((X.shape[0], len(self.classes_)))
        
        for i in range(len(self.classes_)):
            # 对数似然（假设特征独立）
            log_likelihood = np.sum(
                -0.5 * np.log(2 * np.pi * self.vars_[i]) -
                0.5 * ((X - self.means_[i]) ** 2) / self.vars_[i],
                axis=1
            )
            
            # 对数后验 = 对数似然 + 对数先验
            log_posteriors[:, i] = log_likelihood + np.log(self.priors_[i])
        
        return self.classes_[np.argmax(log_posteriors, axis=1)]

print("朴素贝叶斯分类器类定义完成")

In [ ]:
# 测试自定义朴素贝叶斯
naive_bayes_custom = NaiveBayesClassifier()
naive_bayes_custom.fit(X_train_iris, y_train_iris)

naive_bayes_sklearn = GaussianNB()
naive_bayes_sklearn.fit(X_train_iris, y_train_iris)

print("朴素贝叶斯对比:")
print(f"  自定义实现准确率: {naive_bayes_custom.score(X_test_iris, y_test_iris):.4f}")
print(f"  Scikit-learn准确率: {naive_bayes_sklearn.score(X_test_iris, y_test_iris):.4f}")

# 打印参数
print(f"\n自定义朴素贝叶斯参数:")
for i, class_name in enumerate(naive_bayes_custom.classes_):
    print(f"  类别 {class_name}:")
    print(f"    先验概率: {naive_bayes_custom.priors_[i]:.4f}")
    print(f"    均值: {naive_bayes_custom.means_[i][:4].round(2)}")

## 15. 与其他分类器的综合对比

In [ ]:
# 综合对比多种分类器
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# 定义所有分类器
all_classifiers = {
    '逻辑回归': LogisticRegression(random_state=42, max_iter=1000),
    'LDA': LinearDiscriminantAnalysis(),
    'QDA': QuadraticDiscriminantAnalysis(),
    'GaussianNB': GaussianNB(),
    '决策树': DecisionTreeClassifier(random_state=42),
    '随机森林': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'KNN': KNeighborsClassifier()
}

results_all = []

for name, clf in all_classifiers.items():
    try:
        clf.fit(X_train_iris, y_train_iris)
        
        train_score = clf.score(X_train_iris, y_train_iris)
        test_score = clf.score(X_test_iris, y_test_iris)
        cv_scores = cross_val_score(clf, X_iris, y_iris, cv=5)
        
        results_all.append({
            '分类器': name,
            '训练集准确率': train_score,
            '测试集准确率': test_score,
            '交叉验证准确率': cv_scores.mean(),
            '交叉验证标准差': cv_scores.std()
        })
    except Exception as e:
        print(f"{name} 错误: {e}")

# 显示结果
results_all_df = pd.DataFrame(results_all)
results_all_df = results_all_df.sort_values('交叉验证准确率', ascending=False)
print("所有分类器在 Iris 数据集上的性能（按交叉验证准确率排序）:")
print(results_all_df.round(4).to_string(index=False))

In [ ]:
# 可视化性能对比
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 按测试集准确率排序
results_sorted = results_all_df.sort_values('测试集准确率', ascending=True)

# 测试集准确率
y_pos = np.arange(len(results_sorted))
bars = axes[0].barh(y_pos, results_sorted['测试集准确率'], alpha=0.7)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(results_sorted['分类器'])
axes[0].set_xlabel('测试集准确率')
axes[0].set_title('分类器性能对比（测试集）')
axes[0].grid(True, alpha=0.3, axis='x')
axes[0].set_xlim(0.8, 1.05)

# 添加数值标签
for i, (idx, row) in enumerate(results_sorted.iterrows()):
    axes[0].text(row['测试集准确率'], i, f"{row['测试集准确率']:.3f}", 
                va='center', ha='left', fontsize=8)

# 训练集 vs 测试集
axes[1].scatter(results_all_df['训练集准确率'], results_all_df['测试集准确率'], 
           s=100, alpha=0.6)

# 添加对角线
axes[1].plot([0.8, 1.0], [0.8, 1.0], 'r--', label='完美拟合')

# 添加标签
for i, (idx, row) in enumerate(results_all_df.iterrows()):
    axes[1].annotate(row['分类器'], 
                   (row['训练集准确率'], row['测试集准确率']),
                   xytext=(5, 5), textcoords='offset points',
                   fontsize=8, alpha=0.7)

axes[1].set_xlabel('训练集准确率')
axes[1].set_ylabel('测试集准确率')
axes[1].set_title('训练集 vs 测试集准确率')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0.8, 1.05)
axes[1].set_ylim(0.8, 1.05)

plt.tight_layout()
plt.show()

## 16. 判别分析的适用场景

In [ ]:
# 创建不同场景的测试数据
scenarios = {
    '小样本': (make_blobs(n_samples=50, n_features=2, centers=2, random_state=42), '小样本'),
    '大样本': (make_blobs(n_samples=500, n_features=2, centers=2, random_state=42), '大样本'),
    '高维': (make_classification(n_samples=200, n_features=20, n_informative=10, 
                               n_redundant=5, n_classes=2, random_state=42), '高维'),
    '噪声': (make_classification(n_samples=200, n_features=2, n_informative=2, 
                              flip_y=0.2, random_state=42), '高噪声')
}

scenario_results = []

for name, ((X, y), desc) in scenarios.items():
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    methods_scenario = {
        'LDA': LinearDiscriminantAnalysis(),
        'QDA': QuadraticDiscriminantAnalysis(),
        'GaussianNB': GaussianNB()
    }
    
    for method_name, model in methods_scenario.items():
        try:
            model.fit(X_train_s, y_train_s)
            score = model.score(X_test_s, y_test_s)
            scenario_results.append({
                '场景': desc,
                '方法': method_name,
                '测试集准确率': score
            })
        except Exception as e:
            scenario_results.append({
                '场景': desc,
                '方法': method_name,
                '测试集准确率': np.nan
            })

# 显示结果
scenario_df = pd.DataFrame(scenario_results)
scenario_pivot = scenario_df.pivot(index='场景', columns='方法', values='测试集准确率')
print("不同场景下的性能对比:")
print(scenario_pivot.round(4))

# 可视化
plt.figure(figsize=(12, 6))
scenarios_list = list(scenario_pivot.index)
methods_list = list(scenario_pivot.columns)
x = np.arange(len(scenarios_list))
width = 0.25

for i, method in enumerate(methods_list):
    plt.bar(x + i*width, scenario_pivot[method], width, label=method, alpha=0.8)

plt.xlabel('场景')
plt.ylabel('测试集准确率')
plt.title('不同场景下的判别分析性能')
plt.xticks(x + width, scenarios_list)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.ylim(0.6, 1.05)
plt.tight_layout()
plt.show()

## 总结

### 判别分析对比

| 特性 | LDA | QDA | 朴素贝叶斯 |
|------|-----|-----|----------|
| 决策边界 | 线性 | 二次 | 线性（假设独立） |
| 协方差矩阵 | 共享 | 各自不同 | 对角（独立假设） |
| 参数量 | Kp + p² | Kp + Kp² | 2Kp |
| 灵活性 | 低 | 高 | 中等 |
| 过拟合风险 | 低 | 高 | 低 |
| 适用样本量 | 小样本 | 大样本 | 小/大样本 |

### LDA 的优势

- 参数少，不易过拟合
- 计算高效，有闭式解
- 可以同时用于分类和降维
- 适合小样本高维数据

### QDA 的优势

- 更灵活，可以建模更复杂的决策边界
- 适合协方差矩阵差异大的情况
- 在有足够数据时通常优于 LDA

### 朴素贝叶斯的优势

- 计算极其快速
- 特征独立假设简化了计算
- 对缺失值不敏感
- 在某些情况下表现优秀

### 何时使用哪种方法

**使用 LDA 的情况**:
- 样本数量较少
- 各类别的协方差相似
- 需要同时进行分类和降维
- 特征数量较多

**使用 QDA 的情况**:
- 样本数量充足
- 各类别的协方差差异较大
- 决策边界明显非线性
- 需要更高的分类精度

**使用朴素贝叶斯的情况**:
- 特征之间相对独立
- 需要极快的预测速度
- 特征维度很高
- 对实时性要求高

### 实践建议

1. **数据预处理**:
   - 始终标准化数据
   - 处理缺失值
   - 检测并处理异常值

2. **模型选择**:
   - 检查各类的协方差矩阵
   - 使用交叉验证比较 LDA 和 QDA
   - 考虑数据量和维度

3. **参数调优**:
   - LDA: 尝试不同的 shrinkage 值
   - QDA: 使用 reg_param 正则化
   - 使用网格搜索寻找最佳参数

4. **模型评估**:
   - 使用交叉验证
   - 分析混淆矩阵
   - 可视化决策边界

### 常见问题

**Q: LDA 和逻辑回归的区别？**
A: LDA 假设数据服从多元正态分布且各类协方差相同，逻辑回归直接建模条件概率，假设较少。

**Q: 什么时候选择 QDA 而不是 LDA？**
A: 当各类别的分布形状差异明显，且有足够的数据时，QDA 通常更好。

**Q: 朴素贝叶斯为什么叫"朴素"？**
A: 因为它假设特征之间相互独立，这是一个很强的假设，但在实践中往往效果不错。

**Q: LDA 能处理多分类问题吗？**
A: 可以，LDA 原生支持多分类，不需要像逻辑回归那样使用一对多策略。